# miniharness: a study notebook

You built an agent harness in 18 steps, one design pattern at a time. This notebook is the condensed version
to study from: **what each pattern solved, the decisions that mattered, and small runnable examples**.

Every example runs **offline**: `MockProvider` replays scripted model replies, so nothing here calls an API or
costs money, and the outputs are the same every time.

**How to run it** (from the repo root):

```bash
uv run --with jupyter jupyter lab docs/study_notebook.ipynb
```

Or open it in VS Code and pick the project's `.venv` as the kernel.

**Companion files:** `DECISIONS.md` has every decision in full (D1 to D211), and `PATTERNS.md` is your journal,
with a "Your notes" slot under each pattern. This notebook links to both by decision number (e.g. **D63**)
instead of repeating them.

**Contents**
0. The map: stages, patterns, files
1. The architecture in one picture
2. One section per pattern: problem → shape → run it → decisions to remember → check yourself
3. Cross-cutting lessons: the ideas that came up again and again
4. The bugs that real runs found
5. Open decisions (read live from `DECISIONS.md`)
6. Exercises (read live from `PATTERNS.md`)
7. Cheat sheet: which pattern when

## 0. The map

| Stage | Pattern | The problem it solved | Where it lives |
|---|---|---|---|
| 1 | Template Method | one agent loop, customizable steps | `agent.py` (`think`, `act`, `finish` hooks) |
| 2 | Strategy + Adapter | swap LLM providers; two incompatible wire formats | `providers/`, `adapters/` |
| 3 | Factory | `"provider:model"` strings instead of `if/elif` | `providers/registry.py` |
| 4 | Builder | system prompt sections in a cache-friendly order | `prompts.py` |
| 5 | Command | tool calls you can log and undo | `tools/base.py`, `tools/toolbox.py` |
| 6 | Decorator | cost tracking, budget, retry, cache around any provider | `providers/wrappers.py` |
| 7 | Chain of Responsibility | permission rules; model fallbacks | `tools/policy.py`, `FallbackChain` |
| 8 | Proxy | lazy providers; read-only tool views | `providers/proxies.py`, `tools/view.py` |
| 9 | State | circuit breaker; the agent's lifecycle | `providers/breaker.py`, `lifecycle.py` |
| 10 | Observer | watch a run without touching the agent | `events.py`, `observers.py` |
| 11 | Memento + Prototype | checkpoints, rewind (with files), resume, fork | `checkpoint.py`, `Agent.fork()` |
| 12 | Composite | sub-agents as tools | `tools/agent_tool.py` |
| 13 | Mediator | peer agents that talk through a coordinator | `team.py` |
| 14 | Visitor | many operations over a trace tree | `trace.py` |
| 15 | Iterator | streaming tokens and events | `providers/streaming.py`, `Agent.iter_run()` |
| 16 | Facade | one simple entry point over everything | `harness.py`, `providers/stack.py` |
| bonus | Bridge | agent types × providers without N × M classes | `agent_types.py` |
| bonus | Singleton → DI | remove the last global registry | `ProviderRegistry` |

**The working method** (worth as much as the patterns):
1. **Naive first, then refactor.** Stage 2 started from one ugly `call_llm()` with an `if/elif`, so the pain was real before the fix.
2. **Characterization tests before refactoring** (D1). Lock down the exact payloads, then change the structure.
3. **No speculative patterns.** Every pattern needed a concrete problem, and when one didn't have one (Abstract Factory, D14), it waited.
4. **Live runs find what tests can't.** Many of the best decisions came from a real model doing something unexpected (section 4).

## 1. The architecture in one picture

```
                         Harness  (Facade, stage 16)
                            │  builds and wires everything below
          ┌─────────────────┼──────────────────────────────┐
          ▼                 ▼                              ▼
   Agent / PlannerAgent   Toolbox (Invoker)             EventBus ──► ConsolePrinter, JsonlTracer
   ReviewerAgent          ├ PolicyChain (CoR)            (Observer)   └► trace.py visitors (Visitor)
   (Template Method,      ├ Tools → ToolCommand (Command)
    State: lifecycle,     ├ ToolboxView (Proxy)
    Bridge: agent types)  └ AgentTool → sub-Agent (Composite)
          │
          ▼  provider.complete() / provider.stream()   (Strategy; Iterator)
   DiskCache → Retry → BudgetGuard → UsageTracker → FallbackChain → CircuitBreaker / LazyProvider
   (Decorator / Proxy stack, built in order by build_stack())                  (CoR)   (State) (Proxy)
          │
          ▼
   OpenRouterProvider / AnthropicProvider / MockProvider   (made by ProviderRegistry: Factory → DI)
          │
          ▼
   OpenAIChatAdapter / AnthropicMessagesAdapter   (Adapter)  ──►  SDK  ──►  API
```

Two ideas hold it together:
- **Everything is an interface with one or two methods.** `Provider` (`complete`, optionally `stream`) and `ToolSet` (`specs`, `execute`) are small, so wrappers, proxies, views and mocks can stand in for the real thing anywhere.
- **Whoever creates an object wires it** (D204). Objects are passed in (dependency injection) rather than looked up globally.

In [1]:
# Setup: imports and two helpers that script a fake model.
import json, tempfile
from pathlib import Path

from miniharness import Agent, LLMResponse, Message, ToolCall
from miniharness.types import Usage
from miniharness.providers import MockProvider


def say(text, cost=0.001):
    # A scripted model reply with no tool calls.
    return LLMResponse(Message(role="assistant", content=text), "end", Usage(100, 10, cost))


def call(name, cost=0.001, **arguments):
    # A scripted model reply that asks for one tool call.
    msg = Message(role="assistant", tool_calls=[ToolCall(id=f"call-{name}", name=name, arguments=arguments)])
    return LLMResponse(msg, "tool_calls", Usage(100, 10, cost))


tmp = Path(tempfile.mkdtemp())  # a scratch workspace for the file examples
print("ready")

ready


## 2. The patterns

### Stage 1: Template Method

**Problem:** every agent variant runs the same loop (*think → act → repeat*) but differs in the details.

**Shape:** the loop is fixed in the framework, and the steps are hooks a subclass overrides: `initial_messages`, `think`, `act`, `finish` (later also `first_state` and `after_answer`, from the Bridge stage).

**Remember:** inheritance gives you *one* axis of variation per subclass. That limit is why stage 2 used composition (Strategy) for the provider.

In [2]:
class Shouty(Agent):
    def finish(self, response):          # override one step; the loop is untouched
        return response.message.content.upper()

agent = Shouty(MockProvider([call("add", a=2, b=3), say("the answer is 5")]))
print(agent.run("2+3?"))
print([m.role for m in agent.messages])   # system, user, assistant (tool call), tool, assistant

THE ANSWER IS 5
['system', 'user', 'assistant', 'tool', 'assistant']


### Stage 2: Strategy + Adapter

**Problem:** one `call_llm()` function branched on a provider string, rebuilt SDK clients on every call, and buried format translation inline, so it couldn't be tested offline.

**Shape:**
- **Strategy:** a `Provider` protocol with `complete(messages, tools)`. The `Agent` holds one and never asks which.
- **Adapter:** pure `to_wire()` / `from_wire()` translators between *our* `Message`/`ToolCall` types and each wire format.

The Adapter is easiest to see by putting one conversation through both:

In [3]:
from miniharness.adapters import OpenAIChatAdapter, AnthropicMessagesAdapter

convo = [
    Message(role="system", content="Be brief."),
    Message(role="user", content="2*3?"),
    Message(role="assistant", tool_calls=[ToolCall("c1", "multiply", {"a": 2, "b": 3})]),
    Message(role="tool", content="6", tool_call_id="c1"),
]
print("OpenAI / OpenRouter format:")
print(json.dumps(OpenAIChatAdapter().to_wire(convo, {})["messages"], indent=1))
print("\nAnthropic format (system is separate; tool results live in a *user* message):")
wire = AnthropicMessagesAdapter().to_wire(convo, {})
print("system =", wire["system"]); print(json.dumps(wire["messages"], indent=1))

OpenAI / OpenRouter format:
[
 {
  "role": "system",
  "content": "Be brief."
 },
 {
  "role": "user",
  "content": "2*3?"
 },
 {
  "role": "assistant",
  "content": null,
  "tool_calls": [
   {
    "id": "c1",
    "type": "function",
    "function": {
     "name": "multiply",
     "arguments": "{\"a\": 2, \"b\": 3}"
    }
   }
  ]
 },
 {
  "role": "tool",
  "tool_call_id": "c1",
  "content": "6"
 }
]

Anthropic format (system is separate; tool results live in a *user* message):
system = Be brief.
[
 {
  "role": "user",
  "content": "2*3?"
 },
 {
  "role": "assistant",
  "content": [
   {
    "type": "tool_use",
    "id": "c1",
    "name": "multiply",
    "input": {
     "a": 2,
     "b": 3
    }
   }
  ]
 },
 {
  "role": "user",
  "content": [
   {
    "type": "tool_result",
    "tool_use_id": "c1",
    "content": "6"
   }
  ]
 }
]


**Decisions to remember**
- **D1:** characterization tests *first*, with fake SDK clients. The payload assertions never changed during the refactor.
- **D2:** `typing.Protocol` (structural typing). Mocks and fakes qualify without inheriting anything.
- **D3:** adapters are separate from providers, which separates two reasons to change: *format* and *transport*.
- **D8 (still open):** `Message.raw` keeps Anthropic's original content (thinking blocks must be replayed unchanged). It's where the adapter abstraction **leaks**, and every adapter eventually meets something the common interface can't express.

**Check yourself:** why is `Agent(provider=...)` composition rather than `OpenRouterAgent(Agent)` inheritance? *(Answer: provider choice is independent of agent type. With inheritance you'd need a subclass per combination, which is the N × M explosion the Bridge stage named.)*

### Stage 3: Factory

**Problem:** every script turned `"openrouter"` into a provider with its own `if/elif`.

**Shape:** `create_provider("openrouter:qwen/qwen3-coder")`. Provider classes register themselves with a decorator, so adding a provider never edits the factory (the Open/Closed principle).

In [4]:
from miniharness.providers import create_provider, available_providers

print(available_providers())
p = create_provider("mock")
print(type(p).__name__, p.model)
# Only the FIRST colon splits (D15): OpenRouter model ids can contain colons.
print("openrouter:deepseek/deepseek-chat:free".partition(":"))

['anthropic', 'mock', 'openrouter']
MockProvider mock
('openrouter', ':', 'deepseek/deepseek-chat:free')


**Decisions to remember**
- **D12:** self-registering decorator vs one explicit dict. *Magic at a distance*: a provider only exists once its module is imported (D16).
- **D14:** **no Abstract Factory**, deliberately. It builds *families* of matching parts, and there was never a family to keep consistent. It was re-examined in stage 6 and stayed deferred. Knowing when *not* to use a pattern is the lesson.
- **Python note:** a class is already a callable that builds objects, so **a dict of classes *is* a factory**. The decorator only automates filling it.

### Stage 4: Builder

**Problem:** system prompts are assembled from parts, and the **order matters**: prompt caching only reuses an unchanged *prefix*, so volatile facts (date, cwd) must come last.

**Shape:** `SystemPromptBuilder` collects sections in any order, and `build()` validates and renders them in a fixed stable-to-volatile order. `default_prompt()` is the **Director** (a reusable recipe).

In [5]:
from miniharness.prompts import SystemPromptBuilder

prompt = (SystemPromptBuilder()
          .environment(date="2026-10-25")        # called first...
          .instructions("Be brief.")
          .identity("You are a test agent.")
          .build())
print(prompt)                                     # ...but rendered last

You are a test agent.

## Instructions
- Be brief.

## Environment
- date: 2026-10-25


**Decisions to remember**
- **D21/D23:** a Builder earns its place in Python when construction has **rules** (ordering, validation, several output formats), *not* because a constructor has many arguments. Keyword arguments already cover that.
- **D24:** `tool_rules()` takes guidance text, not tool schemas, because the API already sends the schemas.
- **D28:** a *date*, not a timestamp: a coarser value breaks the cache once a day instead of on every call.

### Stage 5: Command

**Problem:** a tool call was a dict lookup plus a function call. Nothing recorded it, and nothing could be undone.

**Shape:** **Command** = `ToolCommand` (one call plus its arguments, result and undo info). **Invoker** = `Toolbox` (runs commands and keeps the history). **Receiver** = the workspace on disk.

In [6]:
from miniharness.tools import Toolbox, default_tools, PolicyChain

box = Toolbox(default_tools(tmp), policy=PolicyChain.allow_all())
print(box.execute(ToolCall("1", "write_file", {"path": "a.txt", "content": "v1"})))
print(box.execute(ToolCall("2", "write_file", {"path": "a.txt", "content": "v2"})))
print(box.execute(ToolCall("3", "write_file", {"path": "../escape.txt", "content": "x"})))  # sandbox (D31)
print("history:", [c.call.name for c in box.history])
box.undo_last(); print("after one undo:", (tmp / "a.txt").read_text())
box.undo_all();  print("after undo_all, exists:", (tmp / "a.txt").exists())

Created a.txt (2 characters)
Overwrote a.txt (2 characters)
Error: PermissionError: Path '../escape.txt' is outside the workspace
history: ['write_file', 'write_file']
after one undo: v1
after undo_all, exists: False


**Decisions to remember**
- **D30:** tools *declare* `side_effects` / `reversible`. A no-op `undo()` on "send email" would lie.
- **D33:** undo by **snapshot** (save the old state, restore it) vs **compensating command** (run the inverse action, as in the Saga pattern). Snapshots are exact for files, and compensation is for things you can't snapshot (payments).
- **D34:** undo **stops** at an irreversible action instead of skipping past it.
- **D35:** undo refuses if the file changed since our write (*optimistic concurrency*).
- **D37:** the default toolbox is read-only (*least privilege*).

**Check yourself:** what makes undo *possible* at all? *(Turning a call into an object. Without the object, there's nothing to hold on to.)*

### Stage 6: Decorator

**Problem:** cost tracking, budgets, retries and caching apply to *every* provider. Building them into each provider duplicates code, and subclassing every combination explodes.

**Shape:** each wrapper takes a Provider, *is* a Provider, and adds one behavior. They stack: `DiskCache(Retry(BudgetGuard(UsageTracker(base))))`.

**The surprise: order is behavior.** Here's the same cache in two positions:

In [7]:
from miniharness.providers import UsageTracker, DiskCache

HI = [Message(role="user", content="hi")]

t1 = UsageTracker(MockProvider([say("x", cost=0.01)]))
outside = DiskCache(t1, tmp / "cache1")          # cache OUTSIDE the tracker
outside.complete(HI, {}); outside.complete(HI, {})
print("cache outside tracker -> spend counted:", t1.total.cost_usd)    # 0.01: a hit is free

t2 = UsageTracker(DiskCache(MockProvider([say("x", cost=0.01)]), tmp / "cache2"))  # cache INSIDE
t2.complete(HI, {}); t2.complete(HI, {})
print("cache inside tracker  -> spend counted:", t2.total.cost_usd)    # 0.02: the hit counted again

cache outside tracker -> spend counted: 0.01
cache inside tracker  -> spend counted: 0.02


**Decisions to remember**
- **D46:** decorators are **composable, but not commutative**. The recommended order is fixed in `build_stack()` (D175).
- **D48:** the budget **fails closed** when a cost is unknown: a budget you can't measure is one you can't enforce.
- **D50:** `Retry` only handles *semantic* failures (such as bad tool JSON), because the SDKs already retry 429s and 5xx errors. Stacking both would mean up to 9 calls.
- **Naming trap (D42):** Python's `@decorator` wraps *functions at definition time*; GoF Decorator wraps *objects at runtime*.

### Stage 7: Chain of Responsibility

**Problem:** several independent rules decide whether a tool call may run, and one `if` ladder would mix them all.

**Shape:** `PolicyChain([...])`. Each policy returns a `Decision` or `None` (pass), and the **first verdict wins**. `ASK` is resolved by an injected approver, and with no approver it becomes a denial.

In [8]:
from miniharness.tools import (AllowReadOnly, AskBeforeSideEffects, DenyTools, MaxCalls, WriteFile, Add)

chain = PolicyChain([DenyTools({"send_email"}), MaxCalls("write_file", 1),
                     AllowReadOnly(), AskBeforeSideEffects()],
                    approver=lambda call: call.arguments.get("path") == "ok.txt")
w, a = WriteFile(tmp), Add()
for c, tool in [(ToolCall("1", "add", {}), a),
                (ToolCall("2", "write_file", {"path": "ok.txt"}), w),
                (ToolCall("3", "write_file", {"path": "ok.txt"}), w)]:
    d = chain.decide(c, tool)
    print(f"{c.name:<11} {d.verdict.value:<6} by {d.policy:<20} {d.reason}")

add         allow  by AllowReadOnly        add is read-only
write_file  allow  by AskBeforeSideEffects write_file was approved
write_file  deny   by MaxCalls             write_file reached its limit of 1 calls


The same pattern over providers: `FallbackChain([main, backup])`. The first provider that answers wins.

In [9]:
from miniharness.providers import FallbackChain

main = MockProvider([ConnectionError("main is down")])
backup = MockProvider([say("answer from backup")])
chain = FallbackChain([main, backup], on=(ConnectionError,))
print(chain.complete(HI, {}).message.content, "| failures:", chain.failures[0][1])

answer from backup | failures: ConnectionError: main is down


**Decisions to remember**
- **CoR vs Decorator (D55):** in a Decorator *every* layer runs. In a chain, the request stops at the first handler that takes it.
- **D57:** `ASK` separates *deciding to ask* (a policy) from *how to ask* (terminal, Slack, UI).
- **D61:** every decision names its reason and the policy that made it, to answer "why was this allowed?"
- **D63, found live:** the first fallback caught *every* API error, so it quietly answered with DeepSeek when the Anthropic key was invalid. **A fallback that catches everything is a bug-hider.** It now falls back only on *transient* errors.

### Stage 8: Proxy

**Problem:** building a provider needs its API key and client up front, even for a fallback that's never used. And some agents should only be able to *see* some tools.

**Shape:** **virtual proxy** `LazyProvider` (build the real provider on first use), and **protection proxy** `ToolboxView` (expose only allowed tools). `BudgetGuard` turned out to be a protection proxy all along (D72).

In [10]:
from miniharness.providers import LazyProvider
from miniharness.tools import ToolboxView

built = []
lazy = LazyProvider(lambda: built.append("built!") or MockProvider([say("hi")]), model="known")
print("model without building:", lazy.model, "| built:", built)
lazy.complete(HI, {}); print("after first call | built:", built)

full = Toolbox(default_tools(tmp), policy=PolicyChain.allow_all())
view = ToolboxView.read_only(full)
print("visible tools:", sorted(view.specs()))
print(view.execute(ToolCall("1", "write_file", {"path": "x", "content": "y"})))  # looks like it doesn't exist (D74)

model without building: known | built: []
after first call | built: ['built!']
visible tools: ['add', 'current_time', 'multiply', 'read_file']
Error: unknown tool 'write_file'


**Decisions to remember**
- **Proxy vs Decorator:** identical structure, different **intent**. Ask whether it changes *what happens* (Decorator) or *whether and when it can happen* (Proxy).
- **D74:** hidden tools look exactly like missing ones (*information hiding*).
- **D75:** defense in depth: the view hides tools *before* the model can try them, and the policy decides *after*.
- **D76:** the Agent depends on `ToolSet` (two methods), not `Toolbox`: the *interface segregation principle*.
- **Live lesson:** with a read-only view, the model never even tried to write. **Hiding is stronger than refusing.**

### Stage 9: State

**Problem:** the fallback retried a dead provider on *every* turn (D65), and the agent's loop had hidden phases.

**Shape:** one class per state, each with its own behavior, returning or setting the next state. `CircuitBreaker`: `closed → open → half_open → closed/open`. Agent: `idle → thinking ⇄ acting_on_tools → done/stopped`.

In [11]:
from miniharness.providers import CircuitBreaker
from miniharness.errors import CircuitOpenError

class Clock:
    now = 0.0
    def __call__(self): return self.now
clock = Clock()

b = CircuitBreaker(MockProvider([ConnectionError(), ConnectionError(), say("recovered")]),
                   threshold=2, cooldown_s=30, clock=clock, on=(ConnectionError,))
for _ in range(2):
    try: b.complete(HI, {})
    except ConnectionError: pass
try: b.complete(HI, {})
except CircuitOpenError as e: print("open:", e)
clock.now = 31
print(b.complete(HI, {}).message.content, "| transitions:", b.transitions)

open: mock is failing; retrying in 30s
recovered | transitions: [('closed', 'open'), ('open', 'half_open'), ('half_open', 'closed')]


In [12]:
# The agent lifecycle, one step at a time
agent = Agent(MockProvider([call("multiply", a=3, b=4), say("12")]))
print(agent.start("3*4?").name)
while not agent.done:
    print(agent.step().name)

thinking
acting_on_tools
thinking
done


**Decisions to remember**
- **State vs Strategy:** both swap an object to change behavior. With Strategy, *you* choose it. With State, **the object changes itself** as things happen.
- **D83:** state-local data. `Closed` owns the failure count, so entering it resets the count for free.
- **D84:** an injected clock (`time.monotonic`), so tests drive time instead of sleeping.
- **D85:** the breaker is a **Proxy by intent and State by implementation**. Patterns combine.

### Stage 10: Observer

**Problem:** to watch the agent work, you had to edit it, or reconstruct events from `agent.messages` afterwards.

**Shape:** the Agent and Toolbox `emit()` typed events on a shared `EventBus`, and subscribers are plain callables. **Emitters don't know who's listening.**

In [13]:
from miniharness.events import EventBus, ToolCalled

bus = EventBus()
seen = []
bus.subscribe(lambda e: seen.append(type(e).__name__))
bus.subscribe(lambda e: 1 / 0)          # a broken observer...
import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    Agent(MockProvider([call("add", a=1, b=1), say("2")]), events=bus).run("1+1")
print(seen)                              # ...doesn't break the run (D95)
print("errors recorded:", len(bus.errors))

['RunStarted', 'StateChanged', 'ModelCalled', 'StateChanged', 'ToolCalled', 'StateChanged', 'ModelCalled', 'StateChanged', 'RunFinished']
errors recorded: 9


**Decisions to remember**
- **D95:** a failing observer is isolated and recorded, so **subjects never depend on their observers**.
- **D97:** events are emitted from framework code, not from overridable hooks, so overriding `think()` keeps them.
- **D104:** **observers can watch but can't veto.** That's why the budget is a proxy, not an observer.
- **D204 (Singleton → DI stage):** a plain `Agent` now wires its own default Toolbox's events, which fixed D98, where `ToolCalled` was silently lost on that path.

### Stage 11: Memento + Prototype

**Problem:** no way to go back (rewind), to survive a stop (resume), or to try alternatives from the same point (fork).

**Shape:** `agent.checkpoint()` returns a deep-copied `Checkpoint`; `restore()` puts everything back, **and undoes file changes made since** (Memento + Command together); `save()` / `load()` persist it as JSON; `fork()` copies the agent (Prototype).

In [14]:
ws = tmp / "rewind"; ws.mkdir()
box = Toolbox(default_tools(ws), policy=PolicyChain.allow_all())
agent = Agent(MockProvider([call("write_file", path="a.txt", content="one"),
                            call("write_file", path="b.txt", content="two"), say("done")]), tools=box)
agent.start("write two files"); agent.step(); agent.step()       # a.txt is written
cp = agent.checkpoint()
while not agent.done: agent.step()                                # b.txt is written
print("before restore:", sorted(p.name for p in ws.iterdir()))
agent.restore(cp)
print("after restore: ", sorted(p.name for p in ws.iterdir()), "| state:", agent.state.name)

class Polite(Agent):
    def finish(self, r): return "Kindly: " + r.message.content
fork = Polite(MockProvider([])).fork(provider=MockProvider([say("hello")]), tools=Toolbox(default_tools()))
print(type(fork).__name__, "<- a fork keeps its subclass (Prototype's real benefit, D115)")

before restore: ['a.txt', 'b.txt']
after restore:  ['a.txt'] | state: thinking
Polite <- a fork keeps its subclass (Prototype's real benefit, D115)


**Decisions to remember**
- **D108:** the memento is a readable dataclass, not a sealed object: JSON needs the fields. Only `Agent` creating and restoring checkpoints is a convention.
- **D112:** rewind checks for irreversible actions *before* undoing anything (all or nothing).
- **D111:** file rewind only works within one process, because the tool history lives in memory.
- **D117 (open):** resuming re-runs pending tool calls, which is *at-least-once* execution.

### Stage 12: Composite

**Problem:** delegating to a sub-agent without its turns filling the parent's context, and without new machinery for policies, events or undo.

**Shape:** `AgentTool(name, description, prototype_agent)` is just a `Tool`. Its `run(task)` drives a fresh copy of an agent whose tools may include more `AgentTool`s: a tree. **The Composite operation:** `side_effects` is computed from the children, recursively.

In [15]:
from miniharness.tools import AgentTool, ReadFile

reader = AgentTool("reader", "reads files", Agent(MockProvider([]), tools=Toolbox([ReadFile(tmp)])))
writer = AgentTool("writer", "writes files", Agent(MockProvider([]), tools=Toolbox([WriteFile(tmp)])))
manager = AgentTool("manager", "delegates", Agent(MockProvider([]), tools=Toolbox([Add(), writer])))
print("reader side effects: ", reader.side_effects)
print("writer side effects: ", writer.side_effects)
print("manager side effects:", manager.side_effects, "<- inherited from its writer child")

helper = AgentTool("helper", "adds", Agent(MockProvider([call("add", a=2, b=3), say("five")])))
parent = Agent(MockProvider([call("helper", task="2+3?"), say("The helper says five.")]), tools=Toolbox([helper]))
print(parent.run("ask"), "| parent messages:", len(parent.messages), "(sub-agent turns stay out)")

reader side effects:  False
writer side effects:  True
manager side effects: True <- inherited from its writer child
The helper says five. | parent messages: 5 (sub-agent turns stay out)


**Decisions to remember**
- **D121:** a read-only sub-agent counts as read-only, so `AllowReadOnly` lets it run without asking.
- **D123:** a `ContextVar` tracks *which agent is running*, so nested events get tagged with `depth` automatically. That's implicit context, invisible in function signatures.
- **D126:** a cycle, or tools that can't be inspected, count as side effects (fail closed).
- **D128:** a depth limit stops runaway recursion.

### Stage 13: Mediator

**Problem:** agents that need to talk *as equals* (a writer and a reviewer). A tree only lets a parent call down.

**Shape:** `Team(members, routes, finishers)`. Members get `send_message(to, text)` (limited by `routes`) and finishers get `approve()`. Members **never reference each other**, and the routing lives in one place.

In [16]:
from miniharness.team import Team

writer = Agent(MockProvider([call("send_message", to="reviewer", text="draft 1"), say("sent"),
                             call("send_message", to="reviewer", text="draft 2"), say("sent")]), tools=Toolbox([]))
reviewer = Agent(MockProvider([call("send_message", to="writer", text="line 2 is weak"), say("sent"),
                               call("approve"), say("ok")]), tools=Toolbox([]))
team = Team({"writer": writer, "reviewer": reviewer},
            routes={"writer": ["reviewer"], "reviewer": ["writer"]}, finishers={"reviewer"})
print("result:", team.run("writer", "Write a haiku."))
for m in team.log: print(f"  {m.sender} -> {m.to}: {m.text}")

result: draft 2
  user -> writer: Write a haiku.
  writer -> reviewer: draft 1
  reviewer -> writer: line 2 is weak
  writer -> reviewer: draft 2


**Decisions to remember: three found in live runs**
- **D136:** with `finish(result)`, a real reviewer "approved" by passing **its own rewrite**. Now `approve()` takes no text, so the result is always a teammate's message. **When a role matters, enforce it in the API, not in the prompt.**
- **D142:** with strict isolation, the reviewer never saw the task, so it couldn't check "exactly 7 words". The mediator now shares the brief: coordination is the mediator's job.
- **D143 (open):** one reviewer approved a 6-word slogan for a 7-word rule, and another miscounted a correct one. **Mechanical rules belong in code, not in an LLM's judgment.**
- **Mediator vs Observer:** Observer *broadcasts* to anyone who subscribed. A Mediator *routes*, knowing every participant and the rules between them.

### Stage 14: Visitor

**Problem:** many operations over the same trace tree (cost per agent, an HTML report, and more later). Making each a method on every node class would bloat the nodes.

**Shape:** nodes only implement `accept(visitor)`, which calls `visitor.visit_<type>(self)` (*double dispatch*). Each operation is a `TraceVisitor` subclass.

In [17]:
from miniharness.observers import JsonlTracer
from miniharness.trace import load_trace, CostSummary, TraceVisitor

trace_file = tmp / "run.jsonl"
bus = EventBus(); bus.subscribe(JsonlTracer(trace_file))
helper = AgentTool("helper", "adds", Agent(MockProvider([call("add", a=2, b=3), say("five", cost=0.002)]),
                                           tools=Toolbox([Add()], events=bus), events=bus))
Agent(MockProvider([call("helper", task="2+3?"), say("five it is")]),
      tools=Toolbox([helper], events=bus), events=bus, name="main").run("ask the helper")

summary = CostSummary(); load_trace(trace_file).accept(summary)
print(summary.table())

class ToolNames(TraceVisitor):         # a brand-new operation: no node class changed
    def __init__(self): self.names = []
    def visit_tool(self, node):
        self.names.append(f"{node.agent}:{node.name}"); super().visit_tool(node)
v = ToolNames(); load_trace(trace_file).accept(v); print(v.names)

agent          calls    tokens in/out        cost  tools  denied    time
main               2         200 / 20   $0.002000      1       0    0.0s
helper             2         200 / 20   $0.003000      1       0    0.0s
total              4         400 / 40   $0.005000      2       0    0.0s
['main:helper', 'helper:add']


**Decisions to remember**
- **The expression problem (D145):** Visitor makes **new operations cheap** and **new node types expensive**. Plain methods on nodes are the opposite. Choose by which one will grow.
- **D148:** events arrive inner-first (a sub-agent finishes before its tool call does), so the builder holds nested runs until their parent's tool call arrives. That relies on tools running one at a time (D39).
- **D154:** the HTML report escapes everything, because tool output is untrusted.
- **Live lesson (D146):** a five-line throwaway visitor found which layer invented two "action items": the *parent*, not the sub-agent.

### Stage 15: Iterator

**Problem:** everything returned only when finished, and the only way to watch a run was to be pushed events.

**Shape:** `provider.stream()` yields `TextDelta`s, then one `Completed(response)`. `agent.iter_run(task)` yields every event as it happens. Inside, each state's `iter_step()` is a **generator**.

**The pattern dissolved into the language:** a Python generator *is* the GoF Iterator class, written for you. The paused stack frame holds the traversal state, and `yield` is `next()`.

In [18]:
from miniharness.providers.streaming import stream_from

for chunk in stream_from(MockProvider([say("streamed answer")]), HI, {}):
    print(type(chunk).__name__, getattr(chunk, "text", ""))   # a provider without stream() is replayed (D157)

agent = Agent(MockProvider([call("add", a=1, b=2), say("3")]))
for event in agent.iter_run("1+2?"):                          # pull, not push
    print(" ", type(event).__name__, getattr(event, "new", "") or getattr(getattr(event, "call", None), "name", ""))

TextDelta streamed 
TextDelta answer
Completed 
  RunStarted 
  StateChanged thinking
  ModelCalled 
  StateChanged acting_on_tools
  ToolCalled add
  StateChanged thinking
  ModelCalled 
  StateChanged done
  RunFinished 


**Decisions to remember**
- **Iterator vs Observer (pull vs push):** the same events, opposite control. With Observer, the subject decides when you hear. With Iterator, *you* decide when it continues, and `break` gives you cancellation for free. `iter_run()` is built *on* the bus (D165).
- **D158:** every Decorator needed a streaming path, because forwarding `stream()` by default would have silently skipped the budget.
- **D159:** "retry" stops meaning anything once the user has seen half an answer, so retry and fallback only switch before the first token.
- **D160, live:** stopping a stream early made that call's cost *unknown*, because usage only arrives at the end. **D164:** a ContextVar held across a `yield` leaks into the caller.

### Stage 16: Facade

**Problem:** a capable agent meant hand-wiring about ten objects in the right order. `examples/hello.py` was 165 lines, mostly wiring.

**Shape:** `Harness(model, budget=..., workspace=..., approve=...)` builds everything from flat keyword arguments with **safe defaults**, then offers `run()`, `follow_up()`, `stream()`, `resume()`, `add_helper()`, `rewind()`, `report()`. Every part stays reachable: `h.agent`, `h.toolbox`, `h.tracker`, `h.events`.

In [19]:
from miniharness.harness import Harness

h = Harness(MockProvider([call("multiply", a=17, b=23), say("391")]), console=False)
print(h.run("17*23?"))
print(h.report())
print("escape hatches:", type(h.agent).__name__, type(h.toolbox).__name__, type(h.tracker).__name__)

391
2 model call(s), 200 in / 20 out tokens, $0.002000
escape hatches: Agent Toolbox UsageTracker


**Decisions to remember**
- **D176:** side effects are **denied unless `approve=` is given**, because a library never calls `input()`. Every run gets a $0.05 budget by default.
- **Facade vs Mediator:** a Mediator sits *between* peers that talk through it. A Facade sits *in front of* a subsystem for an outside client, and the subsystem doesn't know the facade exists.
- **Facade vs Builder:** a Builder assembles one product and is done. A Facade stays as the simple way to *use* many objects. Here the facade *calls* two Directors (`default_prompt()`, `build_stack()`).
- **Live lessons (D179, D182):** the defaults combined in ways the parts didn't. Breaking out of `stream()` left a cost unknown, so the default budget blocked the next call. Moving checkpoint code into the facade exposed an old bug: a helper finishing deleted the main run's checkpoint.

### Bonus: Bridge

**Problem:** the provider side had grown into a whole hierarchy (3 providers, 7 wrappers), but there was only ever one agent class. Adding agent types naively (`OpenRouterPlanner`, `AnthropicPlanner`, ...) is the N × M explosion.

**Shape:** the *abstraction* is `Agent`, the *implementor* is the `Provider` protocol, and composition joins them. `PlannerAgent` (plan first, without tools) and `ReviewerAgent` (draft, then one review round) reshape the loop through hooks. **Not one line changed on the provider side.**

In [20]:
from miniharness.agent_types import PlannerAgent, ReviewerAgent

for cls, script in [(Agent,         [call("add", a=2, b=2), say("4")]),
                    (PlannerAgent,  [say("1. add 2 and 2"), call("add", a=2, b=2), say("4")]),
                    (ReviewerAgent, [call("add", a=2, b=2), say("4"), say("4 (checked)")])]:
    p = MockProvider(script)
    answer = cls(p).run("2+2?")
    print(f"{cls.__name__:<14} {len(p.calls)} model calls -> {answer}")

Agent          2 model calls -> 4
PlannerAgent   3 model calls -> 4
ReviewerAgent  3 model calls -> 4 (checked)


**Decisions to remember**
- **Bridge vs Strategy (D198):** the code shape is the same. The difference is **two hierarchies that both grow**. Until a second agent type existed, this was only a Strategy.
- **D189:** planning is a lifecycle *state*, so it streams, gets traced and checkpoints for free.
- **D197, live:** self-review shares the model's blind spots. Asked for a population figure, the reviewer kept an unverified number and only added a hedge.

### Bonus: Singleton → dependency injection

**Problem:** the last hidden global was the provider registry, a module-level dict filled at import time. Tests had to monkeypatch it (D17), the same pain as `MOCK_RESPONSES` (stage 2) and `TOOLS` (stage 5).

**Shape:** `ProviderRegistry` is a plain class, and `default_registry` is one ordinary instance. Everything that turns a spec into a provider takes `registry=`. **What changed is the direction of the dependency:** the global used to be the only option, and now it's just the default.

In [21]:
from miniharness.providers.registry import default_registry

mine = default_registry.copy()
mine.add("echo", lambda **kw: MockProvider([say("hello from a private provider")], model="echo"))
h = Harness("echo", registry=mine, console=False, budget=None)
print(h.run("hi"))
print("'echo' in the default registry?", "echo" in default_registry)   # untouched: no monkeypatch needed

hello from a private provider
'echo' in the default registry? False


**Decisions to remember**
- **Singleton vs DI:** a Singleton answers "how do I get *the* registry?" with "ask the class". DI answers "you don't get it, you're *given* one". The Singleton's costs are hidden dependencies, shared mutable state, order-dependent tests, and no way to have two.
- **Singleton vs ContextVar (D210):** `agent_scope` stays. It holds *facts about the current execution* (which agent is running), with a separate value per thread or task. **Inject what an object needs to do its job; use a ContextVar for "where am I in the call stack".**
- **Python note:** a module *is* a singleton (imported once), so the Pythonic Singleton is a module-level object, which is exactly what `_REGISTRY` was.
- **D204:** *whoever creates an object wires it.*

## 3. Cross-cutting lessons

These ideas came up again and again. They're more reusable than any single pattern.

**1. Fail closed.** When the harness doesn't know, it refuses: an unknown cost stops the budget (D48), no approver means deny (D57), no policy match means deny (D58), uninspectable side effects count as side effects (D126), and the default toolbox is read-only (D37). Each of those defaults prevents a silent, expensive mistake.

**2. Same structure, different intent.** Several patterns look identical in code (an object holding an object with the same interface). Tell them apart by asking what the wrapper is *for*:

| Pattern | The wrapper... | Example here |
|---|---|---|
| Decorator | *adds* behavior; every layer runs | `UsageTracker`, `Retry`, `DiskCache` |
| Proxy | *controls access*: whether, when, where | `BudgetGuard`, `LazyProvider`, `ToolboxView` |
| Chain of Responsibility | lets *one* handler take the request | `PolicyChain`, `FallbackChain` |
| Strategy | is *chosen by the client*, rarely changes | the `Provider` an `Agent` holds |
| State | *replaces itself* as things happen | `CircuitBreaker` states, lifecycle |
| Bridge | joins *two growing hierarchies* | agent types × providers |

**3. Order is behavior.** The decorator stack (D46), the prompt sections (D23) and the policy chain (D56) all mean something different in a different order. Put the order in one place (`build_stack`, the Builder, the chain list).

**4. Enforce roles in the API, not in the prompt.** Every time a prompt failed to hold a rule, the fix was to make the wrong thing impossible: `approve()` takes no text (D136), hidden tools can't be called (D74), recipients are checked in code (D138).

**5. Patterns compose.** Rewind = Memento + Command. Autosave = Observer driving Memento. Sub-agents = Composite + Prototype + Proxy + Observer. The breaker = Proxy by intent and State by implementation. Real designs are combinations.

**6. Python dissolves some patterns.** Strategy and Command can be plain functions, Iterator is a generator, a dict of classes is a factory, a module is a singleton. Knowing the pattern helps you *recognize* the idiom.

**7. Small interfaces make everything swappable.** `Provider` has one method (plus an optional `stream`), and `ToolSet` has two. That's why mocks, wrappers, proxies and views could stand in anywhere.

## 4. The bugs that real runs found

Tests pass against the behavior you imagined. Live runs show you the behavior you actually built. Each of these was found by running the real thing:

| Found in | What happened | What changed |
|---|---|---|
| Stage 6 | a $0.0000001 budget printed as `$0.0000` | 6 decimal places in the message |
| Stage 7 (D63) | the fallback hid an invalid API key by answering from DeepSeek | fall back only on transient errors |
| Stage 7 (D65) | every turn retried the broken provider first | circuit breaker (stage 9) |
| Stage 10 (D103) | the example reused the variable name `provider` and would crash with `--fallback --cache` | renamed the loop variable |
| Stage 13 (D136) | a reviewer "approved" its own rewrite | `approve()` takes no text |
| Stage 13 (D142) | a reviewer couldn't check rules it never saw | the mediator shares the brief |
| Stage 14 (D146) | an answer listed 4 items when the file had 2 | the trace showed the *parent* invented them |
| Stage 15 (D160) | stopping a stream early made its cost unknown | documented; the budget fails closed |
| Stage 16 (D182) | a helper finishing deleted the main run's checkpoint | autosave only on the main agent's events |
| Environment | `~/Documents` is iCloud-synced and evicted venv and `.git` files: imports hung, a "hidden" flag kept coming back | moved the repo to `~/Developer/repos` |

## 5. Open decisions

Read live from `DECISIONS.md`, so this list stays current as you settle them. Each one is a design question the project deliberately left open: good material for your own next steps.

In [22]:
import re

text = (Path.cwd().parent / "DECISIONS.md" if Path.cwd().name == "docs" else Path("DECISIONS.md")).read_text()
entries = re.findall(r"^### (D\d+)\. (.+?)\n\*\*Status:\*\* (.+?)$", text, flags=re.M)
still_open = [(d, title, status) for d, title, status in entries if status.lower().startswith("open")]
print(f"{len(entries)} decisions, {len(still_open)} still open:\n")
for d, title, status in still_open:
    title = re.sub(r"\s*\((your call|found live|made during implementation)[^)]*\)", "", title)
    print(f"{d:>5}  {title[:95]}")

211 decisions, 26 still open:

   D8  Keep `Message.raw` for Anthropic's replayed content
  D14  No Abstract Factory yet
  D25  `build()` returns a plain string
  D39  Parallel tool calls run sequentially, in the order the model sent them
  D40  History is in memory, unbounded, and shared across runs of the same Toolbox
  D52  Anthropic prices are a hardcoded, simplified table
  D66  Usage of a refusal swallowed by the fallback is invisible to an outer tracker
  D67  After a fallback, one conversation mixes providers
  D73  Code that unwraps wrappers must also see through proxies
  D90  Exceeding `max_steps` still raises `RuntimeError`
  D91  No `AwaitingHuman` state: approval stays synchronous inside the Toolbox
 D100  Events carry live objects; frozen is shallow
 D101  No event for unexpected errors
 D117  Checkpoints are taken between steps; resuming re-runs pending tool calls
 D118  Persisted checkpoints make asynchronous approval possible
 D127  An `AgentTool` call isn't undoable 

## 6. Exercises

Every pattern in `PATTERNS.md` ends with a "Try it". They're collected here (read live from the file). Do one, then write what you learned under that pattern's **Your notes**.

In [23]:
patterns = (Path.cwd().parent / "PATTERNS.md" if Path.cwd().name == "docs" else Path("PATTERNS.md")).read_text()
for section in re.split(r"^## ", patterns, flags=re.M)[1:]:
    name = section.splitlines()[0]
    m = re.search(r"- \*\*Try it:\*\* (.+?)(?=\n- \*\*|\n---|\Z)", section, flags=re.S)
    if m:
        print(f"■ {name}\n  " + " ".join(m.group(1).split()) + "\n")

■ Template Method (stage 1, scaffolded)
  write a `LoggingAgent(Agent)` that overrides only `act()` to print each tool call. The loop itself stays untouched.

■ Builder (stage 4)
  add a `.persona()` section, or decide where `.examples()` (few-shot) belongs in the order and why.

■ Command (stage 5)
  add an `AppendFile` tool. What's its undo info? Is a snapshot or an inverse cheaper here?

■ Decorator (stage 6)
  write a `Timing` wrapper that records the latency of each call. Where in the stack should it go to measure network time only, and where to include the cache?

■ Chain of Responsibility (stage 7)
  write a `DenyPaths` policy that blocks `write_file` outside a `notes/` folder. Where in the chain does it go, and why?

■ Proxy (stage 8)
  write a `LoggingToolSet` that records every call. Is that a proxy or a decorator? Then write one that requires approval. What changed?

■ State (stage 9)
  add a `Paused` agent state that `step()` won't leave until `resume()` is called. What wou

## 7. Cheat sheet: which pattern when

| You notice... | Reach for | Not to be confused with |
|---|---|---|
| the same algorithm, with steps that vary per subclass | **Template Method** | Strategy (composition instead of inheritance) |
| interchangeable implementations picked by the caller | **Strategy** | State (changes itself) |
| an interface that doesn't fit the one you need | **Adapter** | Bridge (designed up front) |
| `if kind == ...: X() elif ...` scattered around | **Factory** (a registry) | Abstract Factory (families of parts) |
| construction with ordering or validation rules | **Builder** (+ Director) | a long constructor (just use kwargs) |
| actions you need to log, queue, replay or undo | **Command** | a plain callback |
| the same extra behavior around many implementations | **Decorator** | Proxy (controls access) |
| several rules, the first that applies decides | **Chain of Responsibility** | Decorator (every layer runs) |
| expensive creation, access control, or remote work | **Proxy** | Decorator (adds behavior) |
| behavior that depends on history (`if self.mode == ...`) | **State** | Strategy |
| "let anyone react when X happens" | **Observer** | Mediator (routes with knowledge) |
| save, restore, rewind | **Memento** | serialization alone (no restore semantics) |
| copy a configured object, keeping its type | **Prototype** | a constructor call |
| trees where a group behaves like a single item | **Composite** | a plain list of children |
| peers wired to each other in a tangle | **Mediator** | Facade (one-way, for outside clients) |
| many operations over a stable set of node types | **Visitor** | methods on the nodes (if types keep growing) |
| results that arrive over time; caller-paced processing | **Iterator** (generators) | Observer (push) |
| a subsystem too complex for its common use | **Facade** | Mediator |
| two hierarchies that both grow | **Bridge** | Strategy (one hierarchy) |
| a global "the one X" that tests must reset | **dependency injection**, not Singleton | ContextVar (per-execution facts) |

---

**Where to go next:** pick an open decision from section 5 that matters for *your* agent work (D39 concurrent tools, D117 exactly-once tools, D143 validators and D8 the leaky `raw` are good candidates), write the decision entry *first*, with its options and trade-offs, then build it. That's the habit this project was really practicing.